# Nathaniel Callabresi Assignment 2

**Course:** GenAI for Software Development (CSCI 455/555)  
**Date:** March 16, 2026  
**Instructor:** Dr. Antonio Mastropaolo  
**TA:** Md. Zahidul Haque Alvi
**Environment:** Python 3.9 (Required)

---

### Project Overview
This notebook implements a complete pipeline for code summarization:
1. **Data Collection:** Scrapes high-quality Java methods from GitHub.
2. **Preprocessing:** Cleans, tokenizes, and extracts JavaDoc summaries.
3. **Embeddings:** Uses CodeT5+ to generate vector representations of code.
4. **Modeling:** Trains a Seq2Seq LSTM model with teacher forcing.
5. **Evaluation:** Validates performance using BLEU, METEOR, BERTScore, and the SIDE metric.

---
## Cell 1: Dependency Setup

In [2]:
# Install dependencies from requirements.txt
!pip3 install -r requirements.txt

# Download WordNet for the METEOR evaluation metric
import nltk
nltk.download('wordnet')

You should consider upgrading via the '/Users/ncallabresi/Documents/wm/c455/assignment2/venv/bin/python3 -m pip install --upgrade pip' command.


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ncallabresi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

---
## Cell 2: Imports and Constants

In [13]:
import os
import glob
import subprocess
import shutil
import javalang
import requests
import json
import random
import torch
import torch.nn as nn
import pandas as pd
import sacrebleu
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel
from tqdm import tqdm
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

# Directory and training constants
CLONE_DIR = "dataset/java_repos"
DATA_DIR = "dataset/summarization"
NUM_PAIRS_REQUIRED = 52000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

---
## Cell 3: Repository Mining

We fetch top Java repositories from GitHub API with the following filters:

- **Language:** Java
- **Stars:** >4000 (to ensure high-quality, well-maintained projects)
- **Forks:** excluded (to avoid duplicate code)

We exclude forked repositories to avoid code duplication, as done in [Tufano et al., 2024](https://arxiv.org/pdf/2402.16480). We use a higher star threshold (>4000 vs >10 in that paper) since our goal is to create quality training data, not to maximize repository coverage.

## Clone Repositories

We clone each repository using `--depth 1` (shallow clone) since we only need the current code snapshot, not the full commit history. This saves time and disk space.

In [4]:
def fetch_top_java_repos(num_repos=50):
    repos = []
    page = 1
    while len(repos) < num_repos:
        url = "https://api.github.com/search/repositories"
        params = {"q": "language:java stars:>4000", "sort": "stars", "order": "desc", "per_page": 100, "page": page}
        response = requests.get(url, params=params)
        for item in response.json().get("items", []):
            if not item.get("fork", False):
                repos.append({"name": item["full_name"], "url": item["clone_url"]})
        page += 1
        if len(repos) >= num_repos:
            break
    return repos[:num_repos]

repos = fetch_top_java_repos(num_repos=70) # 70 repos is more than enough for 50k methods
cloned_repos = []

print("Cloning repositories...")
for repo in tqdm(repos):
    safe_name = repo["name"].replace("/", "_")
    dest = os.path.join(CLONE_DIR, safe_name)
    if not os.path.exists(dest):
        subprocess.run(["git", "clone", "--depth", "1", "--quiet", repo["url"], dest])
    cloned_repos.append(dest)
print(f"Successfully cloned {len(cloned_repos)} repos.")

Cloning repositories...


100%|██████████| 70/70 [00:00<00:00, 3175.03it/s]

Successfully cloned 70 repos.


---
## Cell 4: Method and Summary Extraction

In [5]:
from javalang.tokenizer import tokenize

def contains_non_ascii(text):
    """Check if text contains non-ASCII characters."""
    try:
        text.encode('ascii')
        return False
    except UnicodeEncodeError:
        return True

def count_tokens(source_code):
    """Count the number of Java tokens in source code."""
    try:
        tokens = list(tokenize(source_code))
        return len(tokens)
    except:
        return 0

def extract_code_summary(file_path):
    """Parse a Java file and extract method-summary pairs."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            source_code = f.read()
        tree = javalang.parse.parse(source_code)
    except:
        return []

    pairs = []
    for path, node in tree.filter(javalang.tree.MethodDeclaration):
        if node.body is None:
            continue

        if node.documentation:
            # Clean slashes and asterisks from JavaDocs
            lines = [l.strip().strip('/*').strip() for l in node.documentation.split('\n')]
            summary = " ".join([l for l in lines if l and not l.startswith('@') and not l.startswith('<')])
            summary = summary.split('.')[0].strip().lower() # Get first sentence

            # Extract method body logic
            start_line = node.position.line - 1
            source_lines = source_code.split('\n')
            open_braces = 0
            in_method = False
            method_lines = []
            for line in source_lines[start_line:]:
                method_lines.append(line)
                open_braces += line.count('{')
                open_braces -= line.count('}')
                if '{' in line: in_method = True
                if in_method and open_braces == 0: break

            method_source = " ".join(" ".join(method_lines).split())
            summary = " ".join(summary.split())

            # Apply quality filters
            if not method_source or not summary: continue
            if contains_non_ascii(method_source) or contains_non_ascii(summary): continue
            if count_tokens(method_source) > 128 or len(summary.split()) > 64: continue

            pairs.append((method_source, summary))
    return pairs

all_pairs = []
print("Mining code-summary pairs...")
for repo_dir in tqdm(cloned_repos):
    for root, _, files in os.walk(repo_dir):
        for file in files:
            if file.endswith('.java'):
                all_pairs.extend(extract_code_summary(os.path.join(root, file)))
                if len(all_pairs) >= NUM_PAIRS_REQUIRED: break
        if len(all_pairs) >= NUM_PAIRS_REQUIRED: break
    if len(all_pairs) >= NUM_PAIRS_REQUIRED: break

Mining code-summary pairs...


 21%|██▏       | 15/70 [07:46<28:31, 31.13s/it]  


---
## Cell 5: Shuffle and Save Splits

In [7]:
random.shuffle(all_pairs)

train_pairs = all_pairs[:50000]
val_pairs = all_pairs[50000:51000]
test_pairs = all_pairs[51000:52000]

def save_split(pairs, prefix):
    with open(f"{DATA_DIR}/{prefix}_code.txt", "w", encoding="utf-8") as fc, \
         open(f"{DATA_DIR}/{prefix}_summary.txt", "w", encoding="utf-8") as fs:
        for code, summary in pairs:
            fc.write(code + "\n")
            fs.write(summary + "\n")

save_split(train_pairs, "train")
save_split(val_pairs, "val")
save_split(test_pairs, "test")

---
## Cell 6: Embedding Generation Script

In [8]:
%%writefile get_codet5_embeddings.py
import argparse
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

def main():
    parser = argparse.ArgumentParser(description="embedding using CodeT5+ for LSTM training.")
    parser.add_argument("--input", type=str, required=True)
    parser.add_argument("--output", type=str, required=True)
    parser.add_argument("--max_length", type=int, default=512)
    args = parser.parse_args()

    checkpoint = "Salesforce/codet5p-220m"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

    embedding_matrix = model.encoder.embed_tokens.weight.detach().clone()

    with open(args.input, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    token_ids = []
    for line in tqdm(lines, desc="Tokenizing"):
        ids = tokenizer.encode(line, truncation=True, max_length=args.max_length)
        token_ids.append(ids)

    output = {
        "token_ids": token_ids,
        "embedding_matrix": embedding_matrix,
        "tokenizer_name": checkpoint,
        "vocab_size": embedding_matrix.shape[0],
        "embedding_dim": embedding_matrix.shape[1],
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "bos_token_id": tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.pad_token_id
    }
    torch.save(output, args.output)

if __name__ == "__main__":
    main()

Overwriting get_codet5_embeddings.py


---
## Cell 7: Process Embeddings

In [9]:
# Generate embeddings for training and validation sets.
# Tokenizing code to max 128 length, summaries to 64.
!python3 get_codet5_embeddings.py --input dataset/summarization/train_code.txt --output dataset/summarization/train_code.pt --max_length 128
!python3 get_codet5_embeddings.py --input dataset/summarization/train_summary.txt --output dataset/summarization/train_summary.pt --max_length 64

!python3 get_codet5_embeddings.py --input dataset/summarization/val_code.txt --output dataset/summarization/val_code.pt --max_length 128
!python3 get_codet5_embeddings.py --input dataset/summarization/val_summary.txt --output dataset/summarization/val_summary.pt --max_length 64

/Users/ncallabresi/Documents/wm/c455/assignment2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Tokenizing: 100%|██████████████████████| 50000/50000 [00:04<00:00, 11399.59it/s]
/Users/ncallabresi/Documents/wm/c455/assignment2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Tokenizing: 100%|██████████████████████| 50000/50000 [00:01<00:00, 32404.74it/s]
/Users/ncallabresi/Documents/wm/c455/assignment2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://git

---
## Cell 8: Dataset and LSTM Model

In [10]:
class SummarizationDataset(Dataset):
    def __init__(self, code_pt, summary_pt):
        code_data = torch.load(code_pt, weights_only=False)
        summary_data = torch.load(summary_pt, weights_only=False)
        self.code_ids = code_data["token_ids"]
        self.summary_ids = summary_data["token_ids"]

    def __len__(self): return len(self.code_ids)
    def __getitem__(self, idx): return torch.tensor(self.code_ids[idx]), torch.tensor(self.summary_ids[idx])

# Load metadata for model configuration
meta = torch.load("dataset/summarization/train_code.pt", weights_only=False)
PAD_ID, BOS_ID = meta["pad_token_id"], meta["bos_token_id"]
VOCAB_SIZE, EMBED_DIM = meta["vocab_size"], meta["embedding_dim"]
PRETRAINED_EMBEDDINGS = meta["embedding_matrix"]

def collate_fn(batch):
    codes, summaries = zip(*batch)
    return pad_sequence(codes, batch_first=True, padding_value=PAD_ID), \
           pad_sequence(summaries, batch_first=True, padding_value=PAD_ID)

class LSTMSummarizer(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pretrained_embeddings, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=True, padding_idx=pad_idx)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, src, tgt):
        _, (hidden, cell) = self.encoder(self.dropout(self.embedding(src)))
        output, _ = self.decoder(self.dropout(self.embedding(tgt[:, :-1])), (hidden, cell))
        return self.fc(output)

    def generate(self, src, max_len, bos_id):
        self.eval()
        with torch.no_grad():
            _, (h, c) = self.encoder(self.embedding(src))
            curr_token = torch.tensor([[bos_id]] * src.shape[0], device=src.device)
            outputs = []
            for _ in range(max_len):
                out, (h, c) = self.decoder(self.embedding(curr_token), (h, c))
                curr_token = self.fc(out).argmax(dim=-1)
                outputs.append(curr_token)
            return torch.cat(outputs, dim=1)

model = LSTMSummarizer(VOCAB_SIZE, EMBED_DIM, 256, PRETRAINED_EMBEDDINGS, PAD_ID).to(DEVICE)

---
## Cell 9: Training with Early Stopping

In [11]:
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

train_loader = DataLoader(SummarizationDataset("dataset/summarization/train_code.pt", "dataset/summarization/train_summary.pt"), batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SummarizationDataset("dataset/summarization/val_code.pt", "dataset/summarization/val_summary.pt"), batch_size=64, shuffle=False, collate_fn=collate_fn)

best_bleu1 = 0.0
patience_counter = 0

for epoch in range(10):
    model.train()
    total_loss = 0
    for codes, summaries in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        codes, summaries = codes.to(DEVICE), summaries.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(codes, summaries)
        loss = criterion(outputs.reshape(-1, VOCAB_SIZE), summaries[:, 1:].reshape(-1))
        loss.backward(); optimizer.step(); total_loss += loss.item()

    # Validation
    model.eval(); val_preds, val_refs = [], []
    with torch.no_grad():
        for codes, summaries in val_loader:
            gen_ids = model.generate(codes.to(DEVICE), max_len=20, bos_id=BOS_ID)
            val_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
            val_refs.extend(tokenizer.batch_decode(summaries, skip_special_tokens=True))

    bleu1 = sacrebleu.metrics.BLEU(max_ngram_order=1).corpus_score(val_preds, [val_refs]).score
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | BLEU-1: {bleu1:.2f}")

    if bleu1 > best_bleu1:
        best_bleu1 = bleu1; torch.save(model.state_dict(), "best_lstm_model.pt"); patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 3: break

Epoch 1: 100%|██████████| 782/782 [07:42<00:00,  1.69it/s]


Epoch 1 | Loss: 5.8974 | BLEU-1: 12.37


Epoch 2: 100%|██████████| 782/782 [08:51<00:00,  1.47it/s]


Epoch 2 | Loss: 4.8080 | BLEU-1: 14.91


Epoch 3: 100%|██████████| 782/782 [08:33<00:00,  1.52it/s]


...

Epoch 8 | Loss: 3.5042 | BLEU-1: 20.00


Epoch 9: 100%|██████████| 782/782 [07:15<00:00,  1.79it/s]


Epoch 9 | Loss: 3.3939 | BLEU-1: 20.63


Epoch 10: 100%|██████████| 782/782 [07:53<00:00,  1.65it/s]


Epoch 10 | Loss: 3.2823 | BLEU-1: 20.37


---
## Cell 10: Final Multi-Metric Evaluation:

In [14]:
model.load_state_dict(torch.load("best_lstm_model.pt", weights_only=True))
model.eval()

with open(f"{DATA_DIR}/test_code.txt", "r") as fc, open(f"{DATA_DIR}/test_summary.txt", "r") as fs:
    test_codes_raw = [line.strip() for line in fc]
    test_refs_raw = [line.strip() for line in fs]

test_preds = []
for i in tqdm(range(0, len(test_codes_raw), 64)):
    batch = test_codes_raw[i:i+64]
    encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").input_ids.to(DEVICE)
    gen_ids = model.generate(encoded, max_len=20, bos_id=tokenizer.bos_token_id)
    test_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))

print("\n--- Metrics ---")
for n in range(1, 5):
    print(f"BLEU-{n}: {sacrebleu.metrics.BLEU(max_ngram_order=n).corpus_score(test_preds, [test_refs_raw]).score:.2f}")

m_scores = [meteor_score([r.split()], p.split()) for r, p in zip(test_refs_raw, test_preds)]
print(f"METEOR: {sum(m_scores)/len(m_scores):.4f}")

P, R, F1 = bert_score(test_preds, test_refs_raw, lang="en")
print(f"BERTScore (F1): {F1.mean().item():.4f}")

# SIDE Metric calculation
SIDE_PATH = "./side_model/"
side_tokenizer = AutoTokenizer.from_pretrained(SIDE_PATH)
side_model = AutoModel.from_pretrained(SIDE_PATH).to(DEVICE).eval()

def mean_pooling(model_output, mask):
    token_embeddings = model_output[0]
    mask = mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

side_sims = []
with torch.no_grad():
    for p, c in zip(test_preds, test_codes_raw):
        inputs = side_tokenizer([c, p], padding=True, truncation=True, return_tensors='pt').to(DEVICE)
        outputs = side_model(**inputs)
        embeddings = F.normalize(mean_pooling(outputs, inputs['attention_mask']), p=2, dim=1)
        side_sims.append((embeddings[0] * embeddings[1]).sum().item())
print(f"SIDE Score: {sum(side_sims)/len(side_sims):.4f}")

100%|██████████| 16/16 [00:02<00:00,  6.81it/s]



--- Metrics ---
BLEU-1: 20.80
BLEU-2: 13.31
BLEU-3: 9.20
BLEU-4: 6.72
METEOR: 0.1843


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore (F1): 0.8458
SIDE Score: 0.5446


---
## Cell 11: Export Predictions

In [ ]:
output_file = "assignment2_predictions.json"
results = []
for i in range(len(test_codes_raw)):
    results.append({"id": i, "code": test_codes_raw[i], "reference": test_refs_raw[i], "prediction": test_preds[i]})

with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved {len(results)} predictions to {output_file}")